### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl",
        "val": "/content/val.jsonl",
    },
)

EOS_TOKEN = tokenizer.eos_token or ""

def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        system_msg = ""
        user_msg = ""
        assistant_msg = ""

        for msg in messages:
            role = msg["role"]
            content = msg["content"]

            if role == "system":
                system_msg = content
            elif role == "user":
                user_msg = content
            elif role == "assistant":
                assistant_msg = content

        text = f"""### System:
{system_msg}

### User:
{user_msg}

### Assistant:
{assistant_msg}{EOS_TOKEN}"""
        texts.append(text)

    return {"text": texts}

dataset["train"] = dataset["train"].map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

dataset["val"] = dataset["val"].map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset["val"].column_names,
)

print(dataset["train"][0]["text"])

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

### System:
You are a district traffic coordinator that outputs structured district guidance for RL traffic controllers. Return only valid JSON with fields: strategy, priority_corridor, target_intersections, phase_bias, duration_steps.

### User:
### DISTRICT STATE
city_id: city_0001
district_id: d_09
district_type: commercial
scenario: accident
scenario_type: heavy_rush
decision_step: 0
sim_time: 0
intersection_count: 9
avg_queue: 0.00
max_queue: 0.00
total_queue: 0.00
avg_wait: 0.00
max_wait: 0.00
total_wait: 0.00
avg_outgoing_load: 0.00
max_outgoing_load: 0.00
total_outgoing_load: 0.00
recent_throughput: 0.00
queue_change: 0.00
wait_change: 0.00
throughput_change: 0.00
ns_queue: 0.00
ew_queue: 0.00
ns_wait: 0.00
ew_wait: 0.00
dominant_flow: BALANCED
boundary_queue_total: 0.00
boundary_wait_total: 0.00
spillback_risk: 0
incident_flag: 1
construction_flag: 0
overload_flag: 0
event_flag: 0
top_congested_intersections:
- i_0000 q=0.00 w=0.00 out=0.00 phase=0 boundary=0
- i_0001 q=0.00 w

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [6]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    args=SFTConfig(
        max_seq_length=max_seq_length,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,   # smoke test first
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/80 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.839900
2,1.832400
3,1.846000
4,1.789000
5,1.707400
6,1.465400
7,1.344100
8,1.030000
9,0.842500
10,0.662000


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [9]:
FastLanguageModel.for_inference(model)

system_prompt = (
    "You are a district traffic coordinator that outputs structured district guidance "
    "for RL traffic controllers. Return only valid JSON with fields: "
    "strategy, priority_corridor, target_intersections, phase_bias, duration_steps."
)

user_prompt = """### DISTRICT STATE
city_id: city_0001
district_id: d_05
district_type: commercial
scenario: accident
scenario_type: heavy_rush
decision_step: 60
sim_time: 300
intersection_count: 8
avg_queue: 7.50
max_queue: 14.00
total_queue: 60.00
avg_wait: 1.12
max_wait: 4.00
total_wait: 9.00
avg_outgoing_load: 6.38
max_outgoing_load: 11.00
total_outgoing_load: 51.00
recent_throughput: 18.00
queue_change: 3.00
wait_change: 0.00
throughput_change: -10.00
ns_queue: 60.00
ew_queue: 0.00
ns_wait: 9.00
ew_wait: 0.00
dominant_flow: NS
boundary_queue_total: 60.00
boundary_wait_total: 9.00
spillback_risk: 1
incident_flag: 1
construction_flag: 0
overload_flag: 0
event_flag: 0
top_congested_intersections:
- i_0096 q=14.00 w=2.00 out=11.00 phase=0 boundary=1
- i_0097 q=9.00 w=4.00 out=5.00 phase=1 boundary=1
- i_0086 q=9.00 w=1.00 out=7.00 phase=1 boundary=1"""

inference_prompt = f"""### System:
{system_prompt}

### User:
{user_prompt}

### Assistant:
"""

inputs = tokenizer(inference_prompt, return_tensors="pt").to("cuda")
input_len = inputs["input_ids"].shape[1]

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    do_sample=False,
)

generated_tokens = outputs[:, input_len:]
response = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
print(response)

{"duration_steps": 10, "phase_bias": "NS", "priority_corridor": "NS", "strategy": "incident_response", "target_intersections": ["i_0086", "i_0096", "i_0097"]}


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [10]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")

('llama_lora/tokenizer_config.json',
 'llama_lora/special_tokens_map.json',
 'llama_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
One of the most famous and iconic tall towers in Paris is the Eiffel Tower. Standing at 324 meters (1,063 feet) tall, this wrought iron tower is a symbol of the city and a must-see attraction for tourists from all over the world.<|end_of_text|>

## Evaluating our small fine-tune

In [11]:
import json
from datasets import load_dataset

val_data = load_dataset(
    "json",
    data_files={"val": "/content/val.jsonl"},
)["val"]

def build_inference_prompt(messages):
    system_msg = ""
    user_msg = ""

    for msg in messages:
        if msg["role"] == "system":
            system_msg = msg["content"]
        elif msg["role"] == "user":
            user_msg = msg["content"]

    return f"""### System:
{system_msg}

### User:
{user_msg}

### Assistant:
"""

def generate_response(model, tokenizer, prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    generated_tokens = outputs[:, input_len:]
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True).strip()

for i in range(5):
    sample = val_data[i]
    prompt = build_inference_prompt(sample["messages"])
    pred = generate_response(model, tokenizer, prompt)

    print(f"\n=== EXAMPLE {i} ===")
    print("PREDICTED:")
    print(pred)
    print("\nGROUND TRUTH:")
    print(sample["messages"][-1]["content"])

Generating val split: 0 examples [00:00, ? examples/s]


=== EXAMPLE 0 ===
PREDICTED:
{"duration_steps": 10, "phase_bias": "NS", "priority_corridor": "NS", "strategy": "incident_response", "target_intersections": ["i_0000", "i_0001", "i_0002"]}

GROUND TRUTH:
{"duration_steps": 10, "phase_bias": "NS", "priority_corridor": "NS", "strategy": "incident_response", "target_intersections": ["i_0001", "i_0024", "i_0034"]}

=== EXAMPLE 1 ===
PREDICTED:
{"duration_steps": 10, "phase_bias": "NS", "priority_corridor": "NS", "strategy": "incident_response", "target_intersections": ["i_0006", "i_0007", "i_0015"]}

GROUND TRUTH:
{"duration_steps": 10, "phase_bias": "NONE", "priority_corridor": "arterial", "strategy": "incident_response", "target_intersections": ["i_0016", "i_0027", "i_0038"]}

=== EXAMPLE 2 ===
PREDICTED:
{"duration_steps": 10, "phase_bias": "NS", "priority_corridor": "NS", "strategy": "incident_response", "target_intersections": ["i_0008", "i_0009", "i_0010"]}

GROUND TRUTH:
{"duration_steps": 10, "phase_bias": "NS", "priority_corridor"

In [12]:
import json

REQUIRED_FIELDS = {
    "strategy": str,
    "priority_corridor": str,
    "target_intersections": list,
    "phase_bias": str,
    "duration_steps": int,
}

def validate_output(text):
    try:
        obj = json.loads(text)
    except Exception:
        return {"valid_json": False, "valid_schema": False, "parsed": None}

    for field, expected_type in REQUIRED_FIELDS.items():
        if field not in obj:
            return {"valid_json": True, "valid_schema": False, "parsed": obj}
        if not isinstance(obj[field], expected_type):
            return {"valid_json": True, "valid_schema": False, "parsed": obj}

    return {"valid_json": True, "valid_schema": True, "parsed": obj}

valid_json = 0
valid_schema = 0
n = min(50, len(val_data))

for i in range(n):
    sample = val_data[i]
    prompt = build_inference_prompt(sample["messages"])
    pred = generate_response(model, tokenizer, prompt)
    result = validate_output(pred)

    valid_json += int(result["valid_json"])
    valid_schema += int(result["valid_schema"])

print("JSON validity rate:", valid_json / n)
print("Schema validity rate:", valid_schema / n)

JSON validity rate: 1.0
Schema validity rate: 1.0


In [13]:
def parse_ground_truth(sample):
    return json.loads(sample["messages"][-1]["content"])

field_correct = {k: 0 for k in REQUIRED_FIELDS}
all_correct = 0
count = 0

for i in range(n):
    sample = val_data[i]
    prompt = build_inference_prompt(sample["messages"])
    pred_text = generate_response(model, tokenizer, prompt)
    pred_result = validate_output(pred_text)

    if not pred_result["valid_schema"]:
        continue

    gt = parse_ground_truth(sample)
    pred = pred_result["parsed"]

    sample_all_correct = True
    for field in REQUIRED_FIELDS:
        if pred[field] == gt[field]:
            field_correct[field] += 1
        else:
            sample_all_correct = False

    all_correct += int(sample_all_correct)
    count += 1

print("Evaluated valid-schema samples:", count)
for field in REQUIRED_FIELDS:
    print(f"{field} accuracy:", field_correct[field] / max(count, 1))
print("Exact full-object accuracy:", all_correct / max(count, 1))

Evaluated valid-schema samples: 50
strategy accuracy: 1.0
priority_corridor accuracy: 0.94
target_intersections accuracy: 0.16
phase_bias accuracy: 0.94
duration_steps accuracy: 1.0
Exact full-object accuracy: 0.14
